In [ ]:
import os


project_dir = '/blue/shenhaowang/qingqisong/be-and-active-travel'
os.chdir(project_dir)

In [ ]:
"""
FIX: Patch torch._pytree compatibility issue
Must run this BEFORE importing torchvision or open_clip

"""

import torch
import torch.utils._pytree

# Add the missing function if it doesn't exist
if not hasattr(torch.utils._pytree, 'register_pytree_node'):
    print(f"  Patching torch.utils._pytree (torch version: {torch.__version__})")

    def _dummy_register_pytree_node(*args, **kwargs):
        """No-op placeholder for older torch versions."""
        pass

    torch.utils._pytree.register_pytree_node = _dummy_register_pytree_node
    print("  ✓ Patch applied successfully")
else:
    print(f"  ✓ No patch needed (torch version: {torch.__version__})")

# Verify the patch works by importing torchvision
try:
    import torchvision
    print(f"  ✓ torchvision imported OK (version: {torchvision.__version__})")
except Exception as e:
    print(f"  ✗ torchvision import failed: {e}")

# Check if open_clip works now
try:
    import open_clip
    print(f"  ✓ open_clip imported OK")
    CLIP_AVAILABLE = True
except ImportError:
    print(f"  ✗ open_clip not installed, will use ResNet50 only")
    CLIP_AVAILABLE = False
except Exception as e:
    print(f"  ✗ open_clip import error: {e}")
    CLIP_AVAILABLE = False

print(f"\n  CUDA available: {torch.cuda.is_available()}")

  Patching torch.utils._pytree (torch version: 2.1.2+cu121)
  ✓ Patch applied successfully
  ✓ torchvision imported OK (version: 0.16.2+cu121)
  ✓ open_clip imported OK

  CUDA available: True


In [ ]:
"""
================================================================================
Satellite Image Embeddings: ResNet50 + CLIP

================================================================================
"""

import os
import re
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("  SATELLITE IMAGE EMBEDDING EXTRACTION")
print("=" * 80)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"  Device: {device}")

# ============================================================================
# CONFIG
# ============================================================================

IMAGE_DIR          = './satellite_images_points/'
TRIP_DATASET       = './chicago_trip_level_logistic_dataset.csv'
OUTPUT_EMBEDDINGS  = './satellite_embeddings.csv'
OUTPUT_FINAL       = './chicago_trip_level_with_embeddings.csv'
N_PCA_COMPONENTS   = 20

# ============================================================================
# STEP 1: Parse image filenames
# ============================================================================

print(f"\n{'='*70}")
print("[STEP 1] Parsing image filenames")
print(f"{'='*70}")

pattern = re.compile(
    r'satellite_point(\d+)_([-]?\d+\.?\d*)_([-]?\d+\.?\d*)\.png'
)

records = []
for fp in sorted(Path(IMAGE_DIR).glob('*.png')):
    m = pattern.match(fp.name)
    if m:
        records.append({
            'point_id': int(m.group(1)),
            'lat': float(m.group(2)),
            'lon': float(m.group(3)),
            'filepath': str(fp)
        })

img_df = pd.DataFrame(records)
print(f"  Images found: {len(img_df):,}")

if len(img_df) == 0:
    raise FileNotFoundError(f"No matching images in {IMAGE_DIR}")

print(f"  Sample: {img_df.iloc[0]['filepath']}")

# ============================================================================
# STEP 2A: Extract ResNet50 embeddings (2048-dim)
# ============================================================================

print(f"\n{'='*70}")
print("[STEP 2A] ResNet50 Embeddings (2048-dim)")
print(f"{'='*70}")

resnet = models.resnet50(pretrained=True)
resnet = nn.Sequential(*list(resnet.children())[:-1])
resnet = resnet.to(device)
resnet.eval()

resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

resnet_embeddings = []
resnet_failed = []

for idx, row in img_df.iterrows():
    try:
        img = Image.open(row['filepath']).convert('RGB')
        img_t = resnet_transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            feat = resnet(img_t)
        emb = feat.squeeze().cpu().numpy()
        resnet_embeddings.append(emb)
    except Exception as e:
        resnet_embeddings.append(np.full(2048, np.nan))
        resnet_failed.append(row['filepath'])

    if (idx + 1) % 200 == 0 or (idx + 1) == len(img_df):
        print(f"    {idx+1:,} / {len(img_df):,}")

resnet_array = np.array(resnet_embeddings)
print(f"  ✓ ResNet50 done. Shape: {resnet_array.shape}, Failed: {len(resnet_failed)}")

# Free GPU memory
del resnet
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# ============================================================================
# STEP 2B: Extract CLIP embeddings (512-dim) - if available
# ============================================================================

clip_array = None

try:
    import open_clip

    print(f"\n{'='*70}")
    print("[STEP 2B] CLIP ViT-B/32 Embeddings (512-dim)")
    print(f"{'='*70}")

    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
        'ViT-B-32', pretrained='openai'
    )
    clip_model = clip_model.to(device)
    clip_model.eval()

    clip_embeddings = []
    clip_failed = []

    for idx, row in img_df.iterrows():
        try:
            img = Image.open(row['filepath']).convert('RGB')
            img_t = clip_preprocess(img).unsqueeze(0).to(device)
            with torch.no_grad():
                feat = clip_model.encode_image(img_t)
            feat = feat / feat.norm(dim=-1, keepdim=True)
            emb = feat.squeeze().cpu().numpy().astype(np.float32)
            clip_embeddings.append(emb)
        except Exception as e:
            clip_embeddings.append(np.full(512, np.nan))
            clip_failed.append(row['filepath'])

        if (idx + 1) % 200 == 0 or (idx + 1) == len(img_df):
            print(f"    {idx+1:,} / {len(img_df):,}")

    clip_array = np.array(clip_embeddings)
    print(f"  ✓ CLIP done. Shape: {clip_array.shape}, Failed: {len(clip_failed)}")

    del clip_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

except Exception as e:
    print(f"\n  CLIP not available: {e}")
    print(f"  Continuing with ResNet50 only")

# ============================================================================
# STEP 3: PCA dimensionality reduction
# ============================================================================

print(f"\n{'='*70}")
print("[STEP 3] PCA Dimensionality Reduction")
print(f"{'='*70}")

def apply_pca(emb_array, prefix, n_components=N_PCA_COMPONENTS):
    """Apply PCA to embeddings and return DataFrame with named columns."""
    valid_mask = ~np.isnan(emb_array).any(axis=1)
    emb_valid = emb_array[valid_mask]
    print(f"\n  {prefix}: {emb_array.shape[1]} dims → {n_components} PCs")
    print(f"    Valid: {emb_valid.shape[0]:,} / {emb_array.shape[0]:,}")

    scaler = StandardScaler()
    pca = PCA(n_components=n_components)
    reduced = pca.fit_transform(scaler.fit_transform(emb_valid))

    explained = pca.explained_variance_ratio_.sum() * 100
    print(f"    Variance explained: {explained:.1f}%")

    cols = [f'{prefix}_pc{i+1}' for i in range(n_components)]
    full = np.full((len(emb_array), n_components), np.nan)
    full[valid_mask] = reduced

    return pd.DataFrame(full, columns=cols), pca, scaler

# ResNet PCA
resnet_pca_df, resnet_pca, resnet_scaler = apply_pca(resnet_array, 'resnet')

# CLIP PCA (if available)
clip_pca_df = None
if clip_array is not None:
    clip_pca_df, clip_pca, clip_scaler = apply_pca(clip_array, 'clip')

# Combine all embeddings with coordinates
result_parts = [img_df[['point_id', 'lat', 'lon']].reset_index(drop=True)]
result_parts.append(resnet_pca_df.reset_index(drop=True))
if clip_pca_df is not None:
    result_parts.append(clip_pca_df.reset_index(drop=True))

coord_emb = pd.concat(result_parts, axis=1)
coord_emb.to_csv(OUTPUT_EMBEDDINGS, index=False)

print(f"\n  Saved: {OUTPUT_EMBEDDINGS}")
print(f"  Shape: {coord_emb.shape}")
print(f"  Columns: {coord_emb.columns.tolist()}")

# ============================================================================
# STEP 4: Map embeddings to trip data (origin + destination)
# ============================================================================

print(f"\n{'='*70}")
print("[STEP 4] Mapping embeddings to trips")
print(f"{'='*70}")

trip_df = pd.read_csv(TRIP_DATASET)
print(f"  Trips loaded: {len(trip_df):,}")

# Prepare matching keys (round coordinates)
coord_emb['lat_r'] = coord_emb['lat'].round(6)
coord_emb['lon_r'] = coord_emb['lon'].round(6)
coord_dedup = coord_emb.drop_duplicates(subset=['lat_r', 'lon_r'], keep='first')

trip_df['orig_lat_r'] = trip_df['orig_lat'].round(6)
trip_df['orig_lon_r'] = trip_df['orig_lon'].round(6)
trip_df['dest_lat_r'] = trip_df['dest_lat'].round(6)
trip_df['dest_lon_r'] = trip_df['dest_lon'].round(6)

# Identify all PCA columns
pca_cols = [c for c in coord_emb.columns if '_pc' in c]
print(f"  PCA columns to map: {pca_cols}")

# --- Origin embeddings (O_ prefix) ---
orig_data = coord_dedup[['lat_r', 'lon_r'] + pca_cols].copy()
orig_data = orig_data.rename(columns={c: f'O_{c}' for c in pca_cols})

trip_df = trip_df.merge(
    orig_data, left_on=['orig_lat_r', 'orig_lon_r'],
    right_on=['lat_r', 'lon_r'], how='left'
)
trip_df = trip_df.drop(columns=['lat_r', 'lon_r'], errors='ignore')

# --- Destination embeddings (D_ prefix) ---
dest_data = coord_dedup[['lat_r', 'lon_r'] + pca_cols].copy()
dest_data = dest_data.rename(columns={c: f'D_{c}' for c in pca_cols})

trip_df = trip_df.merge(
    dest_data, left_on=['dest_lat_r', 'dest_lon_r'],
    right_on=['lat_r', 'lon_r'], how='left'
)
trip_df = trip_df.drop(columns=['lat_r', 'lon_r',
                                 'orig_lat_r', 'orig_lon_r',
                                 'dest_lat_r', 'dest_lon_r'], errors='ignore')

# Report
o_cols = [f'O_{c}' for c in pca_cols]
d_cols = [f'D_{c}' for c in pca_cols]
both_ok = (trip_df[o_cols].notna().all(axis=1) & trip_df[d_cols].notna().all(axis=1)).sum()
print(f"  Both O+D matched: {both_ok:,} / {len(trip_df):,} ({both_ok/len(trip_df)*100:.1f}%)")

trip_df.to_csv(OUTPUT_FINAL, index=False)
print(f"  Saved: {OUTPUT_FINAL}")

# ============================================================================
# STEP 5: Logistic Regression Comparison
# ============================================================================

print(f"\n{'='*70}")
print("[STEP 5] Model Comparison: Impact of Image Embeddings")
print(f"{'='*70}")

df = trip_df.copy()

# Feature groups
person_hh = [c for c in [
    'age', 'Female', 'License', 'Employed', 'Student',
    'White', 'Black', 'Asian', 'Hispanic', 'Bachelor_Above',
    'hhveh', 'Zero_Vehicle_HH', 'hhsize', 'Low_Income', 'High_Income',
    'Home_Owner'
] if c in df.columns]

trip_f = [c for c in ['od_dist_mi'] if c in df.columns]

orig_be = [c for c in df.columns
           if c.startswith('O_') and '_pc' not in c and df[c].notna().mean() > 0.5]
dest_be = [c for c in df.columns
           if c.startswith('D_') and '_pc' not in c and df[c].notna().mean() > 0.5]

orig_resnet = [c for c in df.columns if c.startswith('O_resnet_pc')]
dest_resnet = [c for c in df.columns if c.startswith('D_resnet_pc')]
orig_clip   = [c for c in df.columns if c.startswith('O_clip_pc')]
dest_clip   = [c for c in df.columns if c.startswith('D_clip_pc')]

base = person_hh + trip_f

# Define all models to compare
model_configs = {
    'M1: Person/HH only':
        base,
    'M2: + Zone BE (O+D)':
        base + orig_be + dest_be,
    'M3: + ResNet (O+D)':
        base + orig_resnet + dest_resnet,
    'M4: Zone BE + ResNet (O+D)':
        base + orig_be + dest_be + orig_resnet + dest_resnet,
}

# Add CLIP models if available
if orig_clip and dest_clip:
    model_configs['M5: + CLIP (O+D)'] = base + orig_clip + dest_clip
    model_configs['M6: Zone BE + CLIP (O+D)'] = base + orig_be + dest_be + orig_clip + dest_clip
    model_configs['M7: Zone BE + ResNet + CLIP'] = (
        base + orig_be + dest_be + orig_resnet + dest_resnet + orig_clip + dest_clip
    )

# Clean data
all_feats = list(set([f for feats in model_configs.values() for f in feats]))
df_clean = df.dropna(subset=all_feats + ['is_transit']).copy()

for col in person_hh + trip_f:
    df_clean = df_clean[df_clean[col] >= 0]

print(f"\n  Clean obs: {len(df_clean):,} | Transit: {df_clean['is_transit'].mean()*100:.1f}%")

train, test = train_test_split(df_clean, test_size=0.2, random_state=42)
y_train, y_test = train['is_transit'], test['is_transit']
print(f"  Train: {len(train):,} | Test: {len(test):,}")

# Run all models
print(f"\n  {'Model':<40} {'Feats':>6} {'Accuracy':>10} {'AUC':>8}")
print(f"  {'='*68}")

results = {}
for name, feats in model_configs.items():
    feats_ok = [f for f in feats if f in df_clean.columns]
    try:
        lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
        lr.fit(train[feats_ok].values, y_train.values)
        y_prob = lr.predict_proba(test[feats_ok].values)[:, 1]
        y_pred = lr.predict(test[feats_ok].values)
        acc = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)
        print(f"  {name:<40} {len(feats_ok):>6} {acc:>10.4f} {auc:>8.4f}")
        results[name] = {'acc': acc, 'auc': auc, 'n': len(feats_ok)}
    except Exception as e:
        print(f"  {name:<40} FAILED: {e}")

# Improvement analysis
print(f"\n  {'='*68}")
print(f"  KEY COMPARISONS:")
print(f"  {'='*68}")

comparisons = [
    ('M2: + Zone BE (O+D)', 'M4: Zone BE + ResNet (O+D)', 'Adding ResNet to Zone BE'),
]
if 'M6: Zone BE + CLIP (O+D)' in results:
    comparisons.append(
        ('M2: + Zone BE (O+D)', 'M6: Zone BE + CLIP (O+D)', 'Adding CLIP to Zone BE')
    )
if 'M7: Zone BE + ResNet + CLIP' in results:
    comparisons.append(
        ('M2: + Zone BE (O+D)', 'M7: Zone BE + ResNet + CLIP', 'Adding Both to Zone BE')
    )

for base_name, new_name, desc in comparisons:
    if base_name in results and new_name in results:
        b = results[base_name]
        n = results[new_name]
        print(f"\n  {desc}:")
        print(f"    Accuracy: {b['acc']:.4f} → {n['acc']:.4f} ({(n['acc']-b['acc'])/b['acc']*100:+.2f}%)")
        print(f"    AUC:      {b['auc']:.4f} → {n['auc']:.4f} ({(n['auc']-b['auc'])/b['auc']*100:+.2f}%)")

print(f"\n{'='*70}")
print("  COMPLETE!")
print(f"{'='*70}")

  SATELLITE IMAGE EMBEDDING EXTRACTION
  Device: cuda

[STEP 1] Parsing image filenames
  Images found: 1,147
  Sample: satellite_images_points/satellite_point1000_41.855161_-87.732243.png

[STEP 2A] ResNet50 Embeddings (2048-dim)
    200 / 1,147
    400 / 1,147
    600 / 1,147
    800 / 1,147
    1,000 / 1,147
    1,147 / 1,147
  ✓ ResNet50 done. Shape: (1147, 2048), Failed: 0

[STEP 2B] CLIP ViT-B/32 Embeddings (512-dim)


100%|███████████████████████████████████████| 354M/354M [00:04<00:00, 74.7MiB/s]


    200 / 1,147
    400 / 1,147
    600 / 1,147
    800 / 1,147
    1,000 / 1,147
    1,147 / 1,147
  ✓ CLIP done. Shape: (1147, 512), Failed: 0

[STEP 3] PCA Dimensionality Reduction

  resnet: 2048 dims → 20 PCs
    Valid: 1,147 / 1,147
    Variance explained: 57.0%

  clip: 512 dims → 20 PCs
    Valid: 1,147 / 1,147
    Variance explained: 71.4%

  Saved: ./satellite_embeddings.csv
  Shape: (1147, 43)
  Columns: ['point_id', 'lat', 'lon', 'resnet_pc1', 'resnet_pc2', 'resnet_pc3', 'resnet_pc4', 'resnet_pc5', 'resnet_pc6', 'resnet_pc7', 'resnet_pc8', 'resnet_pc9', 'resnet_pc10', 'resnet_pc11', 'resnet_pc12', 'resnet_pc13', 'resnet_pc14', 'resnet_pc15', 'resnet_pc16', 'resnet_pc17', 'resnet_pc18', 'resnet_pc19', 'resnet_pc20', 'clip_pc1', 'clip_pc2', 'clip_pc3', 'clip_pc4', 'clip_pc5', 'clip_pc6', 'clip_pc7', 'clip_pc8', 'clip_pc9', 'clip_pc10', 'clip_pc11', 'clip_pc12', 'clip_pc13', 'clip_pc14', 'clip_pc15', 'clip_pc16', 'clip_pc17', 'clip_pc18', 'clip_pc19', 'clip_pc20']

[STEP 4] Ma

In [ ]:
"""
================================================================================
STEP 0: Save raw embeddings (needed for end-to-end model)

================================================================================
Run this first - it re-extracts and saves the raw 2048/512 dim embeddings
"""

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import re
import os
import warnings
warnings.filterwarnings('ignore')

# Patch torch if needed
if not hasattr(torch.utils._pytree, 'register_pytree_node'):
    torch.utils._pytree.register_pytree_node = lambda *a, **k: None

print("=" * 70)
print("  SAVING RAW EMBEDDINGS (ResNet 2048-dim + CLIP 512-dim)")
print("=" * 70)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
IMAGE_DIR = './satellite_images_points/'

# Parse filenames
pattern = re.compile(r'satellite_point(\d+)_([-]?\d+\.?\d*)_([-]?\d+\.?\d*)\.png')
records = []
for fp in sorted(Path(IMAGE_DIR).glob('*.png')):
    m = pattern.match(fp.name)
    if m:
        records.append({
            'point_id': int(m.group(1)),
            'lat': float(m.group(2)),
            'lon': float(m.group(3)),
            'filepath': str(fp)
        })
img_df = pd.DataFrame(records)
print(f"Images: {len(img_df):,}")

# --- ResNet50 ---
print("\nExtracting ResNet50 (2048-dim)...")
resnet = models.resnet50(pretrained=True)
resnet = nn.Sequential(*list(resnet.children())[:-1])
resnet = resnet.to(device).eval()

resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

resnet_embs = []
for idx, row in img_df.iterrows():
    img = Image.open(row['filepath']).convert('RGB')
    with torch.no_grad():
        feat = resnet(resnet_transform(img).unsqueeze(0).to(device))
    resnet_embs.append(feat.squeeze().cpu().numpy())
    if (idx+1) % 300 == 0:
        print(f"  {idx+1:,} / {len(img_df):,}")

resnet_array = np.array(resnet_embs)
print(f"  Shape: {resnet_array.shape}")

del resnet
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# --- CLIP ---
print("\nExtracting CLIP (512-dim)...")
import open_clip

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
clip_model = clip_model.to(device).eval()

clip_embs = []
for idx, row in img_df.iterrows():
    img = Image.open(row['filepath']).convert('RGB')
    with torch.no_grad():
        feat = clip_model.encode_image(clip_preprocess(img).unsqueeze(0).to(device))
    feat = feat / feat.norm(dim=-1, keepdim=True)
    clip_embs.append(feat.squeeze().cpu().numpy().astype(np.float32))
    if (idx+1) % 300 == 0:
        print(f"  {idx+1:,} / {len(img_df):,}")

clip_array = np.array(clip_embs)
print(f"  Shape: {clip_array.shape}")

del clip_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# --- Save ---
np.save('./raw_resnet_embeddings.npy', resnet_array)
np.save('./raw_clip_embeddings.npy', clip_array)
img_df[['point_id', 'lat', 'lon']].to_csv('./embedding_point_index.csv', index=False)

print(f"\nSaved:")
print(f"  raw_resnet_embeddings.npy: {resnet_array.shape}")
print(f"  raw_clip_embeddings.npy:   {clip_array.shape}")
print(f"  embedding_point_index.csv: {len(img_df)} points")

  SAVING RAW EMBEDDINGS (ResNet 2048-dim + CLIP 512-dim)
Images: 1,147

Extracting ResNet50 (2048-dim)...
  300 / 1,147
  600 / 1,147
  900 / 1,147
  Shape: (1147, 2048)

Extracting CLIP (512-dim)...
  300 / 1,147
  600 / 1,147
  900 / 1,147
  Shape: (1147, 512)

Saved:
  raw_resnet_embeddings.npy: (1147, 2048)
  raw_clip_embeddings.npy:   (1147, 512)
  embedding_point_index.csv: 1147 points


In [ ]:
"""
================================================================================
XGBoost + End-to-End Neural Network Experiments
================================================================================
EXPERIMENT STRUCTURE:

  Experiment 1: XGBoost (nonlinear model, same features)
    → Can a nonlinear model extract more from image embeddings?
    → Compare: LogReg vs XGBoost across all feature configs

  Experiment 2: End-to-End MLP (raw embeddings, no PCA)
    → Let the network learn its own feature extraction
    → Compare: PCA+LogReg vs PCA+XGB vs End-to-End MLP

  Experiment 3: Ablation & Analysis
    → Feature importance from XGBoost
    → Which image dimensions matter most?
    → Do image features help more for certain trip types?
================================================================================
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, roc_auc_score,
                              classification_report, f1_score)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("  XGBOOST + END-TO-END NEURAL NETWORK EXPERIMENTS")
print("=" * 80)

# ============================================================================
# LOAD DATA
# ============================================================================

print("\n[LOADING DATA]")

trip_df = pd.read_csv('./chicago_trip_level_with_embeddings.csv')
print(f"  Trips: {len(trip_df):,}")

# Load raw embeddings
resnet_raw = np.load('./raw_resnet_embeddings.npy')
clip_raw = np.load('./raw_clip_embeddings.npy')
emb_index = pd.read_csv('./embedding_point_index.csv')

print(f"  ResNet raw: {resnet_raw.shape}")
print(f"  CLIP raw:   {clip_raw.shape}")
print(f"  Points:     {len(emb_index):,}")

# ============================================================================
# DEFINE FEATURE GROUPS
# ============================================================================

person_hh = [c for c in [
    'age', 'Female', 'License', 'Employed', 'Student',
    'White', 'Black', 'Asian', 'Hispanic', 'Bachelor_Above',
    'hhveh', 'Zero_Vehicle_HH', 'hhsize', 'Low_Income', 'High_Income',
    'Home_Owner'
] if c in trip_df.columns]

trip_f = [c for c in ['od_dist_mi'] if c in trip_df.columns]
base = person_hh + trip_f

orig_be = [c for c in trip_df.columns
           if c.startswith('O_') and '_pc' not in c and trip_df[c].notna().mean() > 0.5]
dest_be = [c for c in trip_df.columns
           if c.startswith('D_') and '_pc' not in c and trip_df[c].notna().mean() > 0.5]

# PCA embeddings (from previous pipeline)
resnet_pca_o = [f'O_resnet_pc{i+1}' for i in range(20) if f'O_resnet_pc{i+1}' in trip_df.columns]
resnet_pca_d = [f'D_resnet_pc{i+1}' for i in range(20) if f'D_resnet_pc{i+1}' in trip_df.columns]
clip_pca_o = [f'O_clip_pc{i+1}' for i in range(20) if f'O_clip_pc{i+1}' in trip_df.columns]
clip_pca_d = [f'D_clip_pc{i+1}' for i in range(20) if f'D_clip_pc{i+1}' in trip_df.columns]

all_pca = resnet_pca_o + resnet_pca_d + clip_pca_o + clip_pca_d

print(f"\n  Feature groups:")
print(f"    Person/HH + Trip:   {len(base)}")
print(f"    Origin Zone BE:     {len(orig_be)}")
print(f"    Dest Zone BE:       {len(dest_be)}")
print(f"    Image PCA (total):  {len(all_pca)}")

# ============================================================================
# CLEAN DATA
# ============================================================================

all_feats = base + orig_be + dest_be + all_pca
df = trip_df.dropna(subset=all_feats + ['is_transit']).copy()

for col in person_hh + trip_f:
    df = df[df[col] >= 0]

y_col = 'is_transit'
print(f"\n  Clean data: {len(df):,} trips")
print(f"  Transit rate: {df[y_col].mean()*100:.1f}%")
print(f"  Active rate:  {df['is_active'].mean()*100:.1f}%")

# ============================================================================
# EXPERIMENT 1: XGBOOST vs LOGISTIC REGRESSION (Cross-Validation — no leakage)
# ============================================================================

print(f"\n{'='*80}")
print("EXPERIMENT 1: XGBoost vs Logistic Regression")
print(f"{'='*80}")

try:
    from xgboost import XGBClassifier
    print("  XGBoost available ✓")
except ImportError:
    import os
    os.system('pip install xgboost --user --quiet')
    from xgboost import XGBClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y = df[y_col].values

# Model configurations
configs = {
    'C1: Person/HH only':          base,
    'C2: + Zone BE (O+D)':         base + orig_be + dest_be,
    'C3: + PCA img (O+D)':         base + all_pca,
    'C4: Zone BE + PCA img':       base + orig_be + dest_be + all_pca,
    'C5: + ResNet PCA10 (O+D)':    base + resnet_pca_o[:10] + resnet_pca_d[:10],
    'C6: Zone BE + ResNet10':      base + orig_be + dest_be + resnet_pca_o[:10] + resnet_pca_d[:10],
    'C7: + CLIP PCA10 (O+D)':      base + clip_pca_o[:10] + clip_pca_d[:10],
    'C8: Zone BE + CLIP10':        base + orig_be + dest_be + clip_pca_o[:10] + clip_pca_d[:10],
}

print(f"\n  {'Config':<35} {'N':>4}  {'LogReg AUC':>12}  {'XGBoost AUC':>12}  {'Δ':>8}")
print(f"  {'='*75}")

exp1_results = {}

for name, feats in configs.items():
    feats_ok = [f for f in feats if f in df.columns]
    X = df[feats_ok].values

    # Logistic Regression (Pipeline handles scaling inside CV — no leakage)
    lr_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(max_iter=1000, C=0.5, random_state=42))
    ])
    lr_auc = cross_val_score(lr_pipe, X, y, cv=cv, scoring='roc_auc')

    # XGBoost (tree-based, doesn't need scaling — no leakage)
    xgb = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        reg_alpha=0.1, reg_lambda=1.0,
        use_label_encoder=False, eval_metric='logloss',
        random_state=42, verbosity=0
    )
    xgb_auc = cross_val_score(xgb, X, y, cv=cv, scoring='roc_auc')

    delta = xgb_auc.mean() - lr_auc.mean()
    marker = "★" if delta > 0.005 else ""

    print(f"  {name:<35} {len(feats_ok):>4}  "
          f"{lr_auc.mean():.4f}±{lr_auc.std():.3f}  "
          f"{xgb_auc.mean():.4f}±{xgb_auc.std():.3f}  "
          f"{delta:>+.4f} {marker}")

    exp1_results[name] = {
        'n_feat': len(feats_ok),
        'lr_auc_mean': lr_auc.mean(), 'lr_auc_std': lr_auc.std(),
        'xgb_auc_mean': xgb_auc.mean(), 'xgb_auc_std': xgb_auc.std(),
    }

# ============================================================================
# EXPERIMENT 1B: XGBOOST FEATURE IMPORTANCE ANALYSIS
# ============================================================================

print(f"\n{'='*80}")
print("EXPERIMENT 1B: XGBoost Feature Importance (Zone BE + All Image PCA)")
print(f"{'='*80}")

full_feats = [f for f in base + orig_be + dest_be + all_pca if f in df.columns]
X_full = df[full_feats].values

# Train full XGBoost model (interpretive analysis, not generalization eval)
xgb_full = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, verbosity=0
)
xgb_full.fit(X_full, y)

# Feature importance
importance = pd.DataFrame({
    'feature': full_feats,
    'importance': xgb_full.feature_importances_
}).sort_values('importance', ascending=False)

# Categorize features
def categorize_feature(name):
    if name in person_hh:
        return 'Person/HH'
    elif name in trip_f:
        return 'Trip'
    elif name in orig_be or name in dest_be:
        return 'Zone BE'
    elif 'resnet' in name:
        return 'ResNet Image'
    elif 'clip' in name:
        return 'CLIP Image'
    else:
        return 'Other'

importance['category'] = importance['feature'].apply(categorize_feature)

# Print top 30
print(f"\n  Top 30 features by XGBoost importance:")
print(f"  {'Rank':>4} {'Feature':<35} {'Importance':>10} {'Category':<15}")
print(f"  {'-'*70}")
for i, (_, row) in enumerate(importance.head(30).iterrows(), 1):
    print(f"  {i:>4} {row['feature']:<35} {row['importance']:>10.4f} {row['category']:<15}")

# Category-level importance
print(f"\n  Importance by Category:")
cat_imp = importance.groupby('category')['importance'].agg(['sum', 'mean', 'count'])
cat_imp = cat_imp.sort_values('sum', ascending=False)
print(f"  {'Category':<15} {'Total Imp':>10} {'Mean Imp':>10} {'N Features':>10}")
print(f"  {'-'*50}")
for cat, row in cat_imp.iterrows():
    print(f"  {cat:<15} {row['sum']:>10.4f} {row['mean']:>10.4f} {int(row['count']):>10}")

# ============================================================================
# EXPERIMENT 2: END-TO-END NEURAL NETWORK
# ============================================================================

print(f"\n{'='*80}")
print("EXPERIMENT 2: End-to-End Neural Network")
print(f"{'='*80}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"  Device: {device}")

# --- Build raw embedding lookup ---
emb_index['lat_r'] = emb_index['lat'].round(6)
emb_index['lon_r'] = emb_index['lon'].round(6)

coord_to_idx = {}
for idx_row, row in emb_index.iterrows():
    key = (round(row['lat'], 6), round(row['lon'], 6))
    coord_to_idx[key] = idx_row

# Map trips to embedding indices
df['orig_emb_idx'] = df.apply(
    lambda r: coord_to_idx.get((round(r['orig_lat'], 6), round(r['orig_lon'], 6)), -1),
    axis=1
)
df['dest_emb_idx'] = df.apply(
    lambda r: coord_to_idx.get((round(r['dest_lat'], 6), round(r['dest_lon'], 6)), -1),
    axis=1
)

valid_mask = (df['orig_emb_idx'] >= 0) & (df['dest_emb_idx'] >= 0)
df_e2e = df[valid_mask].copy()
print(f"  Trips with raw embeddings: {len(df_e2e):,} / {len(df):,}")

# --- Define end-to-end model ---
class TransitChoiceNet(nn.Module):
    """
    End-to-end model: tabular + raw image embeddings → transit choice.
    """

    def __init__(self, n_tabular, n_img, img_hidden=64, img_out=32,
                 mlp_hidden=128, dropout=0.3):
        super().__init__()

        self.origin_img_net = nn.Sequential(
            nn.Linear(n_img, img_hidden),
            nn.BatchNorm1d(img_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(img_hidden, img_out),
            nn.ReLU()
        )

        self.dest_img_net = nn.Sequential(
            nn.Linear(n_img, img_hidden),
            nn.BatchNorm1d(img_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(img_hidden, img_out),
            nn.ReLU()
        )

        total_in = n_tabular + img_out * 2
        self.classifier = nn.Sequential(
            nn.Linear(total_in, mlp_hidden),
            nn.BatchNorm1d(mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x_tab, x_img_o, x_img_d):
        o_feat = self.origin_img_net(x_img_o)
        d_feat = self.dest_img_net(x_img_d)
        combined = torch.cat([x_tab, o_feat, d_feat], dim=1)
        return self.classifier(combined).squeeze(-1)


def train_e2e_model(X_tab_train, X_img_o_train, X_img_d_train, y_train,
                     X_tab_test, X_img_o_test, X_img_d_test, y_test,
                     n_img_dim, n_epochs=50, lr=0.001, batch_size=256,
                     model_name=""):
    """Train end-to-end model and return test AUC."""

    tab_train = torch.FloatTensor(X_tab_train).to(device)
    img_o_train = torch.FloatTensor(X_img_o_train).to(device)
    img_d_train = torch.FloatTensor(X_img_d_train).to(device)
    y_tr = torch.FloatTensor(y_train).to(device)

    tab_test = torch.FloatTensor(X_tab_test).to(device)
    img_o_test = torch.FloatTensor(X_img_o_test).to(device)
    img_d_test = torch.FloatTensor(X_img_d_test).to(device)

    train_ds = TensorDataset(tab_train, img_o_train, img_d_train, y_tr)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = TransitChoiceNet(
        n_tabular=X_tab_train.shape[1],
        n_img=n_img_dim,
        img_hidden=64 if n_img_dim <= 512 else 128,
        img_out=32,
        mlp_hidden=128,
        dropout=0.3
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)
    criterion = nn.BCEWithLogitsLoss()

    best_auc = 0
    best_acc = 0
    best_epoch = 0
    patience_counter = 0

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        for batch_tab, batch_o, batch_d, batch_y in train_loader:
            optimizer.zero_grad()
            logits = model(batch_tab, batch_o, batch_d)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        model.eval()
        with torch.no_grad():
            test_logits = model(tab_test, img_o_test, img_d_test)
            test_probs = torch.sigmoid(test_logits).cpu().numpy()

        test_auc = roc_auc_score(y_test, test_probs)
        test_pred = (test_probs >= 0.5).astype(int)
        test_acc = accuracy_score(y_test, test_pred)

        scheduler.step(epoch_loss)

        if test_auc > best_auc:
            best_auc = test_auc
            best_acc = test_acc
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= 10:
            break

        if (epoch + 1) % 10 == 0:
            print(f"    {model_name} Epoch {epoch+1}: loss={epoch_loss/len(train_loader):.4f} "
                  f"AUC={test_auc:.4f} Acc={test_acc:.4f}")

    print(f"    {model_name} Best: Epoch {best_epoch+1}, AUC={best_auc:.4f}, Acc={best_acc:.4f}")
    return best_auc, best_acc


# --- Prepare data for E2E models ---
tabular_feats = base + orig_be + dest_be
tabular_feats = [f for f in tabular_feats if f in df_e2e.columns]
pca_feats = [f for f in all_pca if f in df_e2e.columns]

y_all = df_e2e[y_col].values

# Get raw image embeddings for each trip
orig_indices = df_e2e['orig_emb_idx'].values.astype(int)
dest_indices = df_e2e['dest_emb_idx'].values.astype(int)

X_resnet_o = resnet_raw[orig_indices]
X_resnet_d = resnet_raw[dest_indices]
X_clip_o = clip_raw[orig_indices]
X_clip_d = clip_raw[dest_indices]

X_both_o = np.concatenate([X_resnet_o, X_clip_o], axis=1)
X_both_d = np.concatenate([X_resnet_d, X_clip_d], axis=1)

print(f"\n  Tabular features:  {len(tabular_feats)}")
print(f"  PCA features:      {len(pca_feats)}")
print(f"  ResNet O/D:        {X_resnet_o.shape} each")
print(f"  CLIP O/D:          {X_clip_o.shape} each")
print(f"  Combined O/D:      {X_both_o.shape} each")

# --- FIXED: Single train/test split, scaler fit only on training set ---
train_idx, test_idx = train_test_split(
    np.arange(len(df_e2e)), test_size=0.2,
    stratify=y_all, random_state=42
)

# Standardize tabular features — fit ONLY on training set
X_tab_raw = df_e2e[tabular_feats].values
X_pca_raw = df_e2e[pca_feats].values

scaler_tab = StandardScaler()
X_tab_scaled = np.zeros_like(X_tab_raw, dtype=np.float64)
X_tab_scaled[train_idx] = scaler_tab.fit_transform(X_tab_raw[train_idx])
X_tab_scaled[test_idx] = scaler_tab.transform(X_tab_raw[test_idx])

# For models that use tabular + PCA image features
scaler_tab_pca = StandardScaler()
X_tab_pca_raw = np.column_stack([X_tab_raw, X_pca_raw])
X_tab_pca_scaled = np.zeros_like(X_tab_pca_raw, dtype=np.float64)
X_tab_pca_scaled[train_idx] = scaler_tab_pca.fit_transform(X_tab_pca_raw[train_idx])
X_tab_pca_scaled[test_idx] = scaler_tab_pca.transform(X_tab_pca_raw[test_idx])

y_tr = y_all[train_idx]
y_te = y_all[test_idx]

print(f"\n  Train: {len(train_idx):,}  Test: {len(test_idx):,}")

# --- Run E2E experiments with multiple random seeds ---
print(f"\n  Running End-to-End experiments (3 seeds each)...")

e2e_configs = {
    'E2E: Tab only (no image)': {
        'img_o': np.zeros((len(df_e2e), 1)),
        'img_d': np.zeros((len(df_e2e), 1)),
        'n_img': 1
    },
    'E2E: Tab + ResNet raw (2048)': {
        'img_o': X_resnet_o,
        'img_d': X_resnet_d,
        'n_img': 2048
    },
    'E2E: Tab + CLIP raw (512)': {
        'img_o': X_clip_o,
        'img_d': X_clip_d,
        'n_img': 512
    },
    'E2E: Tab + Both raw (2560)': {
        'img_o': X_both_o,
        'img_d': X_both_d,
        'n_img': 2560
    },
}

print(f"\n  {'Config':<40} {'AUC (mean±std)':>20} {'Acc (mean±std)':>20}")
print(f"  {'='*80}")

e2e_results = {}

for name, cfg in e2e_configs.items():
    aucs = []
    accs = []

    for seed in [42, 123, 456]:
        torch.manual_seed(seed)
        np.random.seed(seed)

        auc, acc = train_e2e_model(
            X_tab_train=X_tab_scaled[train_idx],
            X_img_o_train=cfg['img_o'][train_idx],
            X_img_d_train=cfg['img_d'][train_idx],
            y_train=y_tr,
            X_tab_test=X_tab_scaled[test_idx],
            X_img_o_test=cfg['img_o'][test_idx],
            X_img_d_test=cfg['img_d'][test_idx],
            y_test=y_te,
            n_img_dim=cfg['n_img'],
            n_epochs=60,
            lr=0.001,
            batch_size=256,
            model_name=f"{name}(seed={seed})"
        )
        aucs.append(auc)
        accs.append(acc)

    auc_mean, auc_std = np.mean(aucs), np.std(aucs)
    acc_mean, acc_std = np.mean(accs), np.std(accs)

    print(f"\n  {name:<40} {auc_mean:.4f}±{auc_std:.4f}    {acc_mean:.4f}±{acc_std:.4f}")

    e2e_results[name] = {
        'auc_mean': auc_mean, 'auc_std': auc_std,
        'acc_mean': acc_mean, 'acc_std': acc_std,
        'aucs': aucs, 'accs': accs
    }

# ============================================================================
# EXPERIMENT 3: COMPREHENSIVE COMPARISON TABLE
# ============================================================================

print(f"\n\n{'='*80}")
print("EXPERIMENT 3: COMPREHENSIVE COMPARISON")
print(f"{'='*80}")

# --- LogReg baselines (using properly scaled data) ---
lr_base = LogisticRegression(max_iter=1000, C=0.5, random_state=42)
lr_base.fit(X_tab_scaled[train_idx], y_tr)
lr_base_auc = roc_auc_score(y_te, lr_base.predict_proba(X_tab_scaled[test_idx])[:, 1])
lr_base_acc = accuracy_score(y_te, lr_base.predict(X_tab_scaled[test_idx]))

lr_pca = LogisticRegression(max_iter=1000, C=0.5, random_state=42)
lr_pca.fit(X_tab_pca_scaled[train_idx], y_tr)
lr_pca_auc = roc_auc_score(y_te, lr_pca.predict_proba(X_tab_pca_scaled[test_idx])[:, 1])
lr_pca_acc = accuracy_score(y_te, lr_pca.predict(X_tab_pca_scaled[test_idx]))

# --- XGBoost baselines (tree-based, use unscaled data) ---
xgb_base = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                          use_label_encoder=False, eval_metric='logloss', verbosity=0,
                          random_state=42)
xgb_base.fit(X_tab_raw[train_idx], y_tr)
xgb_base_auc = roc_auc_score(y_te, xgb_base.predict_proba(X_tab_raw[test_idx])[:, 1])
xgb_base_acc = accuracy_score(y_te, xgb_base.predict(X_tab_raw[test_idx]))

xgb_pca = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                          use_label_encoder=False, eval_metric='logloss', verbosity=0,
                          random_state=42)
xgb_pca.fit(X_tab_pca_raw[train_idx], y_tr)
xgb_pca_auc = roc_auc_score(y_te, xgb_pca.predict_proba(X_tab_pca_raw[test_idx])[:, 1])
xgb_pca_acc = accuracy_score(y_te, xgb_pca.predict(X_tab_pca_raw[test_idx]))

# --- Print comprehensive table ---
print(f"\n  {'Method':<45} {'Features':<25} {'AUC':>8} {'Acc':>8}")
print(f"  {'='*90}")
print(f"  {'--- Logistic Regression ---':<45}")
print(f"  {'LR: Tabular only':<45} {'Zone BE + Person/HH':<25} {lr_base_auc:>8.4f} {lr_base_acc:>8.4f}")
print(f"  {'LR: Tabular + PCA img':<45} {'+ PCA20 embeddings':<25} {lr_pca_auc:>8.4f} {lr_pca_acc:>8.4f}")

print(f"\n  {'--- XGBoost ---':<45}")
print(f"  {'XGB: Tabular only':<45} {'Zone BE + Person/HH':<25} {xgb_base_auc:>8.4f} {xgb_base_acc:>8.4f}")
print(f"  {'XGB: Tabular + PCA img':<45} {'+ PCA20 embeddings':<25} {xgb_pca_auc:>8.4f} {xgb_pca_acc:>8.4f}")

print(f"\n  {'--- End-to-End MLP ---':<45}")
for name, r in e2e_results.items():
    feat_desc = 'Tab only' if 'no image' in name else name.split(':')[1].strip()
    print(f"  {name:<45} {feat_desc:<25} {r['auc_mean']:>8.4f} {r['acc_mean']:>8.4f}")

# ============================================================================
# EXPERIMENT 4: SUBGROUP ANALYSIS (FIXED: only evaluate on test set)
# ============================================================================

print(f"\n\n{'='*80}")
print("EXPERIMENT 4: Does image help more for certain subgroups?")
print(f"{'='*80}")

# Add distance category
df_e2e['dist_cat'] = pd.cut(
    df_e2e['od_dist_mi'],
    bins=[0, 1, 3, 5, 100],
    labels=['<1mi', '1-3mi', '3-5mi', '>5mi']
)

# Add CBD proximity category
if 'O_dist_cbd_mi' in df_e2e.columns:
    cbd_median = df_e2e.iloc[train_idx]['O_dist_cbd_mi'].median()  # median from train only
    df_e2e['cbd_cat'] = np.where(df_e2e['O_dist_cbd_mi'] <= cbd_median, 'Near CBD', 'Far CBD')

# Helper: get test-set indices for a boolean mask over df_e2e
def get_test_subgroup(mask_over_df_e2e):
    """Given a boolean mask over df_e2e, return the test-set rows."""
    mask_arr = mask_over_df_e2e.values if hasattr(mask_over_df_e2e, 'values') else mask_over_df_e2e
    # test_idx are positions in df_e2e; check which test positions satisfy the mask
    sub_test_pos = [i for i in test_idx if mask_arr[i]]
    return np.array(sub_test_pos)

print(f"\n  --- By Trip Distance ---")
print(f"  {'Distance':<12} {'N_test':>6} {'Transit%':>9}  "
      f"{'LR(tab)':>8} {'LR(+img)':>9} {'XGB(tab)':>9} {'XGB(+img)':>10}")
print(f"  {'-'*75}")

for cat in ['<1mi', '1-3mi', '3-5mi', '>5mi']:
    mask = (df_e2e['dist_cat'] == cat)
    sub_pos = get_test_subgroup(mask)

    if len(sub_pos) < 50:
        continue

    sub_y = y_all[sub_pos]
    if len(np.unique(sub_y)) < 2:
        continue

    n = len(sub_pos)
    transit_pct = sub_y.mean() * 100

    # LR tabular (scaled)
    lr_t_auc = roc_auc_score(sub_y, lr_base.predict_proba(X_tab_scaled[sub_pos])[:, 1])
    # LR + img (scaled)
    lr_i_auc = roc_auc_score(sub_y, lr_pca.predict_proba(X_tab_pca_scaled[sub_pos])[:, 1])
    # XGB tabular (raw)
    xgb_t_auc = roc_auc_score(sub_y, xgb_base.predict_proba(X_tab_raw[sub_pos])[:, 1])
    # XGB + img (raw)
    xgb_i_auc = roc_auc_score(sub_y, xgb_pca.predict_proba(X_tab_pca_raw[sub_pos])[:, 1])

    print(f"  {cat:<12} {n:>6} {transit_pct:>8.1f}%  "
          f"{lr_t_auc:>8.4f} {lr_i_auc:>9.4f} {xgb_t_auc:>9.4f} {xgb_i_auc:>10.4f}")

if 'cbd_cat' in df_e2e.columns:
    print(f"\n  --- By CBD Proximity ---")
    print(f"  {'Location':<12} {'N_test':>6} {'Transit%':>9}  "
          f"{'LR(tab)':>8} {'LR(+img)':>9} {'XGB(tab)':>9} {'XGB(+img)':>10}")
    print(f"  {'-'*75}")

    for cat in ['Near CBD', 'Far CBD']:
        mask = (df_e2e['cbd_cat'] == cat)
        sub_pos = get_test_subgroup(mask)

        if len(sub_pos) < 50:
            continue

        sub_y = y_all[sub_pos]
        if len(np.unique(sub_y)) < 2:
            continue

        n = len(sub_pos)
        transit_pct = sub_y.mean() * 100

        lr_t_auc = roc_auc_score(sub_y, lr_base.predict_proba(X_tab_scaled[sub_pos])[:, 1])
        lr_i_auc = roc_auc_score(sub_y, lr_pca.predict_proba(X_tab_pca_scaled[sub_pos])[:, 1])
        xgb_t_auc = roc_auc_score(sub_y, xgb_base.predict_proba(X_tab_raw[sub_pos])[:, 1])
        xgb_i_auc = roc_auc_score(sub_y, xgb_pca.predict_proba(X_tab_pca_raw[sub_pos])[:, 1])

        print(f"  {cat:<12} {n:>6} {transit_pct:>8.1f}%  "
              f"{lr_t_auc:>8.4f} {lr_i_auc:>9.4f} {xgb_t_auc:>9.4f} {xgb_i_auc:>10.4f}")

# ============================================================================
# EXPERIMENT 5: PREDICTING ACTIVE TRAVEL (FIXED: proper scaling per target)
# ============================================================================

print(f"\n\n{'='*80}")
print("EXPERIMENT 5: Multiple Dependent Variables")
print(f"{'='*80}")

for target, label in [('is_transit', 'Transit'), ('is_active', 'Active Travel'),
                       ('is_walk', 'Walking'), ('is_auto', 'Auto')]:
    if target not in df_e2e.columns:
        continue

    y_target = df_e2e[target].values
    if len(np.unique(y_target)) < 2:
        continue

    rate = y_target.mean() * 100

    # Use a fresh stratified split for each target to ensure proper stratification
    tr_idx_t, te_idx_t = train_test_split(np.arange(len(df_e2e)), test_size=0.2,
                                           stratify=y_target, random_state=42)

    y_tr_t = y_target[tr_idx_t]
    y_te_t = y_target[te_idx_t]

    # Scale tabular features — fit only on training set for this target
    scaler_t = StandardScaler()
    X_tab_tr_t = scaler_t.fit_transform(X_tab_raw[tr_idx_t])
    X_tab_te_t = scaler_t.transform(X_tab_raw[te_idx_t])

    scaler_tp = StandardScaler()
    X_tab_pca_tr_t = scaler_tp.fit_transform(X_tab_pca_raw[tr_idx_t])
    X_tab_pca_te_t = scaler_tp.transform(X_tab_pca_raw[te_idx_t])

    # LR tabular
    lr1 = LogisticRegression(max_iter=1000, C=0.5, random_state=42)
    lr1.fit(X_tab_tr_t, y_tr_t)
    lr1_auc = roc_auc_score(y_te_t, lr1.predict_proba(X_tab_te_t)[:, 1])

    # LR + img
    lr2 = LogisticRegression(max_iter=1000, C=0.5, random_state=42)
    lr2.fit(X_tab_pca_tr_t, y_tr_t)
    lr2_auc = roc_auc_score(y_te_t, lr2.predict_proba(X_tab_pca_te_t)[:, 1])

    # XGB tabular (no scaling needed)
    xgb1 = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8,
                           use_label_encoder=False, eval_metric='logloss', verbosity=0,
                           random_state=42)
    xgb1.fit(X_tab_raw[tr_idx_t], y_tr_t)
    xgb1_auc = roc_auc_score(y_te_t, xgb1.predict_proba(X_tab_raw[te_idx_t])[:, 1])

    # XGB + img (no scaling needed)
    xgb2 = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8,
                           use_label_encoder=False, eval_metric='logloss', verbosity=0,
                           random_state=42)
    xgb2.fit(X_tab_pca_raw[tr_idx_t], y_tr_t)
    xgb2_auc = roc_auc_score(y_te_t, xgb2.predict_proba(X_tab_pca_raw[te_idx_t])[:, 1])

    img_gain_lr = lr2_auc - lr1_auc
    img_gain_xgb = xgb2_auc - xgb1_auc

    print(f"\n  {label} (rate={rate:.1f}%):")
    print(f"    LR:  Tab={lr1_auc:.4f}  +Img={lr2_auc:.4f}  Δ={img_gain_lr:+.4f}")
    print(f"    XGB: Tab={xgb1_auc:.4f}  +Img={xgb2_auc:.4f}  Δ={img_gain_xgb:+.4f}")


# ============================================================================
# FINAL ANALYSIS
# ============================================================================



  XGBOOST + END-TO-END NEURAL NETWORK EXPERIMENTS

[LOADING DATA]
  Trips: 20,815
  ResNet raw: (1147, 2048)
  CLIP raw:   (1147, 512)
  Points:     1,147

  Feature groups:
    Person/HH + Trip:   17
    Origin Zone BE:     10
    Dest Zone BE:       10
    Image PCA (total):  80

  Clean data: 13,013 trips
  Transit rate: 17.9%
  Active rate:  18.1%

EXPERIMENT 1: XGBoost vs Logistic Regression
  XGBoost available ✓

  Config                                 N    LogReg AUC   XGBoost AUC         Δ
  C1: Person/HH only                    17  0.8045±0.011  0.9195±0.003  +0.1150 ★
  C2: + Zone BE (O+D)                   37  0.8644±0.006  0.9348±0.004  +0.0704 ★
  C3: + PCA img (O+D)                   97  0.8414±0.010  0.9330±0.003  +0.0916 ★
  C4: Zone BE + PCA img                117  0.8610±0.007  0.9337±0.003  +0.0727 ★
  C5: + ResNet PCA10 (O+D)              37  0.8319±0.011  0.9330±0.003  +0.1010 ★
  C6: Zone BE + ResNet10                57  0.8649±0.006  0.9345±0.003  +0.0696 ★
  C7